In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../../data/processed/laptop_train_processed.csv")
df.head()

,aspect_context,label
0,night and skip taking the [ASPECT] with me bec...,1
1,me because of the good [ASPECT],2
2,tech guy then said the [ASPECT] does not do 1-...,0
3,direct my concern to the [ASPECT] which is the...,0
4,the [ASPECT] then said the service center,1


In [3]:
# only keep model inputs

X = df['aspect_context']
y = df['label']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
# it is preprocessed in such a way that each word is separated by single-space

def tokenize(text):
    return text.split()

In [6]:
train_tokens = X_train.apply(tokenize)
train_tokens.head()

1495    [and, the, screen, on, this, thing, is, absolu...
1551    [they, were, shipping, out, bad, [ASPECT], and...
1829    [another, [ASPECT], that, is, laughable, is, the]
1269    [as, a, die-hard, [ASPECT], enthusiast,, i, sh...
1277    [any, laptop, not, purchased, with, [ASPECT], ...
Name: aspect_context, dtype: object

In [7]:
# collect only unique tokens in training set

vocab = set()

for tokens in train_tokens:
    vocab.update(tokens)

vocab = sorted(vocab)
len(vocab)

2959

# Computing TF-IDF for each tokens

In [8]:
train_tokens.iloc[0]

['and',
 'the',
 'screen',
 'on',
 'this',
 'thing',
 'is',
 'absolutely',
 'amazing',
 'for',
 'high',
 'quality',
 'videos',
 'and',
 'movies',
 'and',
 'gaming.']

In [9]:
# this computes document frequency for each word in the vocabulary
# number of documents in which the word appears


doc_freq = {}

for word in vocab:
    doc_freq[word] = sum(word in tokens for tokens in train_tokens)

In [10]:
# compute IDF for each word


N = len(train_tokens)

idf = {
    word: np.log(N / (1 + doc_freq[word]))
    for word in vocab
}

In [11]:
# TF-IDF computation

# below is how it works
# compute TF-IDF vector for a given token list
# [tf-idf(w) = tf(w) * idf(w) for each word w in vocab]



def compute_tfidf(tokens, vocab, idf):
    tfidf_vector = np.zeros(len(vocab))
    token_counts = Counter(tokens)
    total_tokens = len(tokens)
    
    for i, word in enumerate(vocab):
        tf = token_counts[word] / total_tokens if total_tokens > 0 else 0
        tfidf_vector[i] = tf * idf[word]
    
    return tfidf_vector

In [12]:
# create TF-IDF matrix for training set

X_train_tfidf = np.array([compute_tfidf(tokens, vocab, idf) for tokens in train_tokens])

X_train_tfidf.shape

(1850, 2959)

In [13]:
# transform test set

test_tokens = X_test.apply(tokenize)

X_test_tfidf = np.array([compute_tfidf(tokens, vocab, idf) for tokens in test_tokens])
X_test_tfidf.shape

(463, 2959)

In [14]:
X_train_tfidf

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(1850, 2959))

In [19]:
# checking once if it is right

[i for i in zip(
    np.where(X_train_tfidf[0] != 0)[0][:10],
    [vocab[i] for i in np.where(X_train_tfidf[0] != 0)[0][:10]]
)]


[(np.int64(192), 'absolutely'),
 (np.int64(262), 'amazing'),
 (np.int64(273), 'and'),
 (np.int64(1058), 'for'),
 (np.int64(1108), 'gaming.'),
 (np.int64(1236), 'high'),
 (np.int64(1381), 'is'),
 (np.int64(1689), 'movies'),
 (np.int64(1804), 'on'),
 (np.int64(2067), 'quality')]